# 4. Reliable RAG

Add guard rails: grade retrieved chunks, generate, then verify the answer is grounded.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Basic RAG trusts whatever retrieval returns. If the top-k contains irrelevant chunks the model may still use them (garbage in, garbage out), and even with good chunks it may add unsupported claims. **Reliable RAG** adds two checkpoints: a *relevance grader* before generation and a *groundedness check* after.

## How it works

1. Retrieve more candidates than you need (k=6)
2. LLM grades each chunk: relevant yes/no, drop the no's
3. If nothing survives, refuse with 'I could not find...'
4. Generate from the surviving chunks
5. LLM verifies every claim is supported; retry or refuse if not

```text
 question -> retrieve k=6 -> [grade relevance] -> keep relevant
                                  | none?  -> "I could not find relevant information"
                                  v
                            generate -> [grounded?] --no--> retry / refuse
                                  | yes
                                  v
                               answer
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

## Step 1: grade every retrieved chunk
Only chunks that the LLM judges relevant go forward.

In [ ]:
def is_relevant(query, chunk):
    return llm_yes(f"Question: {query}\n\nPassage: {chunk}\n\nDoes the passage contain information useful for answering the question?")

def grounded(answer_text, contexts):
    ctx = "\n".join(contexts)
    return llm_yes(f"Context:\n{ctx}\n\nAnswer: {answer_text}\n\nIs every claim in the answer supported by the context?")

## Step 2: the reliable pipeline
If nothing relevant is found, or the answer is not grounded, we refuse instead of hallucinating.

In [ ]:
def reliable_rag(query, k=6, max_retries=1):
    hits = store.search(query, k)
    good = [h["text"] for h in hits if is_relevant(query, h["text"])]
    if not good:
        return "I could not find relevant information.", []
    for _ in range(max_retries + 1):
        response = answer(query, good)
        if grounded(response, good):
            return response, good
    return "I could not produce a well-grounded answer.", good

for q in ["What is Reciprocal Rank Fusion?", "What is the capital of Australia?"]:
    print("Q:", q, "\nA:", reliable_rag(q)[0], "\n")

## Watch the graders work
Print every grading decision so you can see what the pipeline filtered out and why.

In [ ]:
q = "What is Reciprocal Rank Fusion?"
for h in store.search(q, 6):
    print("relevant" if is_relevant(q, h["text"]) else "dropped ", f"{h['score']:.3f}", h["text"][:90].replace("\n", " "))

## Plain RAG vs Reliable RAG on questions that should be refused
The golden questions are answerable; these are not. A reliable system should say it cannot answer.

In [ ]:
import pandas as pd
unanswerable = ["Who won the football world cup in 2018?", "What is the refund policy of the AeroBook laptop?", "How tall is Mount Everest?"]
rows = []
for q in unanswerable:
    plain, _ = rag(q, store, k=3)
    reliable, _ = reliable_rag(q)
    rows.append({"question": q, "plain RAG": plain[:110], "reliable RAG": reliable[:110]})
pd.DataFrame(rows)

## Cost of reliability
Every guard is an extra LLM call. Measure it - this is the price you pay.

In [ ]:
import time
q = "How many dimensions does the all-MiniLM-L6-v2 model produce?"
t = time.time(); rag(q, store, k=3); plain_t = time.time() - t
t = time.time(); reliable_rag(q); rel_t = time.time() - t
print(f"plain: {plain_t:.1f}s   reliable: {rel_t:.1f}s   ({rel_t / max(plain_t, 1e-9):.1f}x slower)")

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Fewer hallucinations and off-topic answers | 2-8x more LLM calls and latency |
| Explicit 'I don't know' behaviour | A weak grader model produces false positives/negatives |
| Each guard is simple and individually testable | Over-strict grading can refuse answerable questions |

## When to use it

Customer-facing or high-stakes answers where a wrong answer is worse than a refusal (support, legal, medical, finance).

## Experiments to try

- Grade all chunks in one prompt instead of one call each and compare speed.
- Return which claims failed the groundedness check.
- Add a confidence score and let the UI show it.

## Key takeaways

- Grade before generating, verify after.
- Refusing is a feature.
- Guards cost latency - use a small model for them.

---
**Next:** [5. Choose Chunk Size](../05_choose_chunk_size/choose_chunk_size.ipynb)